In [ ]:
# Install packages
!pip install -q transformers sentencepiece

import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import torch
from tqdm import tqdm

print("="*60)
print("AKKADIAN TRANSLATION - NLLB INFERENCE")
print("="*60)

# Configuration
MAX_LENGTH = 256
MODEL_PATH = "/kaggle/input/datasets/jennyli13/ver7-8-nllb-600m-model/ver7.8_final-nllb-model"

# Load model
print("\n[1/3] Loading fine-tuned NLLB model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Device: {device}")
print("✅ Model loaded successfully!")

# Load test data
print("\n[2/3] Loading test data...")
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(f"Test samples: {len(test_df)}")

# Translate function
def translate_batch(texts, batch_size=16):
    translations = []
    
    tokenizer.src_lang = "eng_Latn"
    target_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")  # ADD: forced_bos_token_id
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", max_length=MAX_LENGTH, 
                          truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,  # ADD: tell NLLB to generate English
                max_length=MAX_LENGTH,
                num_beams=10,                         # ADD: was default (4)
                length_penalty=1.2,                   # ADD: encourages slightly longer output
                no_repeat_ngram_size=3,               # ADD: reduces repetition
                early_stopping=True,                  # ADD: stop when all beams finish
            )
        
        translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    
    return translations

# Generate predictions
print("\n[3/3] Generating predictions...")
predictions = translate_batch(test_df['transliteration'].tolist())

# Create submission
submission_df = pd.DataFrame({
    'id': test_df['id'], 
    'translation': predictions
})

submission_df.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("✅ DONE! submission.csv created")
print("="*60)
print(f"Rows: {len(submission_df)}")
print("\nFirst 3 predictions:")
print(submission_df.head(3))

AKKADIAN TRANSLATION - NLLB INFERENCE

[1/3] Loading fine-tuned NLLB model...


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Device: cuda
✅ Model loaded successfully!

[2/3] Loading test data...
Test samples: 4

[3/3] Generating predictions...


Translating: 100%|██████████| 1/1 [00:05<00:00,  5.58s/it]


✅ DONE! submission.csv created
Rows: 4

First 3 predictions:
   id                                        translation
0   0  Thus the Kanesh colony: For the ... they charg...
1   1  In the City assembly there shall be no credito...
2   2  Since you are the executive arm for me, he has...
